## 라이브러리 설치 및 구글 드라이브 연동


In [ ]:
# 필수 라이브러리 설치 (Qwen-VL 공식 지원 패키지 포함)
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate qwen-vl-utils torch torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 모델

In [ ]:
import torch
import json
from PIL import Image # pillow 라이브러리
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor # 모델 로딩
from qwen_vl_utils import process_vision_info

# ==========================================
# 1. 모델 & 프로세서 로드 (T4 GPU 메모리에 맞게 bfloat16 사용-> bfloat지원x, float16으로 수정 1109_11pm)
# ==========================================
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

print("모델 로딩 중... (약 2~3분 소요)")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto" # 모델이 RAM 할당 알아서 해줌
)
processor = AutoProcessor.from_pretrained(model_id)
print("모델 로딩 완료!")

# ==========================================
# 2. 누끼 이미지 전처리 (투명 배경 -> 회색 변환)
# ==========================================
original_image_path = "/content/drive/MyDrive/image/sample_image.png"
processed_image_path = "processed_image.jpg" # VLM에 넣을 회색 배경 처리된 이미지

try:
    img = Image.open(original_image_path)

    # 분석할 때만 간단하게 이미지 크기 줄임
    # 원본 해상도 -> 512x512
    img.thumbnail((512, 512))

    # 이미지에 알파(투명) 채널이 있는지 확인
    # Pillow 이미지 모드 - Luminance/Alpha
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        img = img.convert("RGBA")

        # VLM이 인식하기 가장 좋은 중립 회색(128, 128, 128) 배경 캔버스 생성
        gray_bg = Image.new("RGB", img.size, (128, 128, 128))

        # 회색 캔버스 위에 원본 이미지 얹기 (알파 채널을 마스크로 사용)
        # R=[0], G=[1], B=[2], A=[3] - 투명도 alpha
        gray_bg.paste(img, mask=img.split()[3])

        # JPG로 임시 저장 (VLM 인풋용)
        gray_bg.save(processed_image_path, format="JPEG")
        print(f"전처리 완료: 투명 배경을 회색으로 변환하여 {processed_image_path}에 저장했습니다.")
    else:
        # 이미 투명 배경이 없는 경우 원본 그대로 사용
        processed_image_path = original_image_path
        print("투명 배경이 없는 이미지입니다. 원본을 그대로 사용합니다.")
except FileNotFoundError:
    print(f"에러: {original_image_path} 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드했는지 확인해주세요.")


# ==========================================
# 3. 시스템 프롬프트 세팅
# ==========================================
system_prompt = """당신은 동네 식당과 소상공인의 매출을 올려주는 탁월한 마케팅 전략가입니다.
제공된 음식 사진(회색 배경으로 전처리됨)과 상황 설명을 분석하여 홍보 전략을 세우고, 이미지 생성 AI가 쓸 영문 프롬프트와 이미지 위에 별도로 얹을 한국어 광고 문구를 작성하세요.
답변은 반드시 아래 구조의 JSON 형식으로만 출력하고 다른 설명은 절대 덧붙이지 마세요.

{
  "sd_prompt": "영문 배경 묘사, 반드시 realistic soft drop shadow under the food, natural directional lighting 키워드를 포함할 것",
  "negative_prompt": "text, words, watermark, typography, floating object, bad proportions, unnatural shadows",
  "ad_copy": "한국어 마케팅 카피 (짧고 강렬하게)",
  "strategy_reason": "전략적 의도 설명",
  "aspect_ratio": "1:1 또는 4:5 또는 9:16 중 택 1"
}"""

# 사용자 input
user_text = "비 와서 손님 없어. 남은 빵 팔고 싶어"

# ==========================================
# 4. 프롬프트 포맷팅 및 추론 실행
# ==========================================
# 대화 기록 세팅
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": [
        # 원본 PNG가 아닌 '회색 배경으로 전처리된 이미지'를 VLM에 전달
        # 512x512 사이즈로 제한
        {"type": "image", "image": processed_image_path, "max_pixels": 262144},
        {"type": "text", "text": user_text}
    ]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages) # 이미지 경로 정보

# tensor 변환
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

print("\nVLM 추론 중...")
# 최대 토큰 개수 - 512개 / 사용자 질문 + AI 답변
generated_ids = model.generate(**inputs, max_new_tokens=512)
# AI 답변만 trim
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

# 디코딩: 숫자 -> 글자
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

# ==========================================
# 5. 결과 파싱 및 출력
# ==========================================
print("\n--- JSON 파싱 결과 ---")
try:
    clean_json_str = output_text.strip().replace("```json", "").replace("```", "").strip()
    result_dict = json.loads(clean_json_str) # 글자 -> 파이썬 데이터
    for key, value in result_dict.items():
        print(f"[{key}]: {value}")
except json.JSONDecodeError:
    print("JSON 파싱 실패! 원본 텍스트:\n", output_text)